In [1]:
%load_ext autoreload
%autoreload 2

In [17]:
from core.nn.timesfm import NewsTimesFM_2p5_Model
from transformers import AutoTokenizer
from safetensors.torch import load_file
from transformers.models.modernbert import ModernBertModel, ModernBertConfig
import torch
import polars as pl
from core.training.data.dataset import TimesFMDataset, collate_fn
from core.nn.text_encoder.tokenizer import NewsTokenizerWrapper
from torch.utils.data import DataLoader
from functools import partial
import plotly.express as px
import random

In [3]:
tokenizer = AutoTokenizer.from_pretrained("/home/pomelk1n/Workspace/finam-forecast/models/newstimesfm-2p5-small-bert")
news_tokenizer = NewsTokenizerWrapper(tokenizer=tokenizer)
news_timesfm = NewsTimesFM_2p5_Model.from_pretrained("/home/pomelk1n/Workspace/finam-forecast/models/newstimesfm-2p5-small-bert", dtype=torch.bfloat16)

Tokenizer max length 1000000000000000019884624838656 is not equal to requested max_length 8192. Setting to 8192.
Tokenizer truncation side right is not left. Setting to left.
Missing keys : ['stacked_xf.0.attn.rotary_position_embedding.max_seq_len', 'stacked_xf.0.cross_attn.ts_rotary_position_embedding.max_seq_len', 'stacked_xf.0.cross_attn.text_rotary_position_embedding.max_seq_len', 'stacked_xf.1.attn.rotary_position_embedding.max_seq_len', 'stacked_xf.1.cross_attn.ts_rotary_position_embedding.max_seq_len', 'stacked_xf.1.cross_attn.text_rotary_position_embedding.max_seq_len', 'stacked_xf.2.attn.rotary_position_embedding.max_seq_len', 'stacked_xf.2.cross_attn.ts_rotary_position_embedding.max_seq_len', 'stacked_xf.2.cross_attn.text_rotary_position_embedding.max_seq_len', 'stacked_xf.3.attn.rotary_position_embedding.max_seq_len', 'stacked_xf.3.cross_attn.ts_rotary_position_embedding.max_seq_len', 'stacked_xf.3.cross_attn.text_rotary_position_embedding.max_seq_len', 'stacked_xf.4.attn.ro

In [5]:
ds = TimesFMDataset(
    "/home/pomelk1n/Workspace/finam-forecast/dataset/FinamDataset/FinamDataset/train.arrow",news_tokenizer
)

In [6]:
dl = DataLoader(ds, batch_size=1, collate_fn=partial(collate_fn, text_pad_token_id=news_tokenizer.tokenizer.pad_token_id, text_model_max_length=news_tokenizer.tokenizer.model_max_length))

In [7]:
iter_dl = iter(dl)

In [12]:
batch = next(iter_dl)

In [13]:
inputs_ts = batch["inputs_ts"].to(torch.bfloat16)
mask_ts = batch["mask_ts"].to(torch.bfloat16)

inputs_text = batch["inputs_text"]
mask_text = batch["mask_text"]

output = news_timesfm.forecast(
    inputs_ts=inputs_ts, 
    mask_ts=mask_ts,
    inputs_text=inputs_text,
    mask_text=mask_text,
)
output

tensor([[[67.5000, 67.5000, 67.5000,  ..., 68.5000, 68.0000, 68.0000],
         [69.5000, 70.0000, 69.0000,  ..., 71.0000, 70.0000, 69.5000],
         [67.5000, 68.0000, 67.5000,  ..., 70.0000, 68.5000, 69.0000],
         ...,
         [23.3750, 33.0000, 18.7500,  ..., 51.2500, 42.0000, 41.7500],
         [23.7500, 31.3750, 17.0000,  ..., 44.2500, 37.2500, 36.5000],
         [27.2500, 34.5000, 22.6250,  ..., 48.0000, 42.0000, 43.2500]]],
       dtype=torch.bfloat16, grad_fn=<AddBackward0>)

In [10]:
batch["targets"]

tensor([[[23.3600, 22.4400, 23.2000,  ..., 32.2700, 32.3900, 32.7900],
         [25.0600, 25.2400, 25.1800,  ..., 39.0900, 39.5100, 39.9800],
         [25.4600, 26.0800, 26.7200,  ..., 41.4300, 42.0900, 41.8500],
         ...,
         [38.6700, 38.5100, 37.8200,  ..., 57.7100, 58.1300, 55.0800],
         [37.4500, 37.8000, 38.0100,  ..., 53.7800, 52.3300, 56.7300],
         [38.0700, 37.7200, 37.6100,  ..., 46.6500, 46.4500, 49.5300]]])

In [15]:
loss_fn = torch.nn.MSELoss()

outpatch_len = output.shape[-1]
total_loss = loss_fn(output, batch["targets"].to(torch.bfloat16))
loss_temp = 0.0
for i in range(outpatch_len):
    loss = loss_fn(output[..., i], batch["targets"][..., i].to(torch.bfloat16))
    loss_temp += loss
    print(f"Step {i}, Loss: {loss.item()}")
total_loss, loss_temp / outpatch_len

Step 0, Loss: 3.5625
Step 1, Loss: 53.75
Step 2, Loss: 27.875
Step 3, Loss: 79.5
Step 4, Loss: 136.0
Step 5, Loss: 146.0
Step 6, Loss: 137.0
Step 7, Loss: 142.0
Step 8, Loss: 128.0
Step 9, Loss: 81.5
Step 10, Loss: 123.0
Step 11, Loss: 107.0
Step 12, Loss: 70.0
Step 13, Loss: 119.5
Step 14, Loss: 129.0
Step 15, Loss: 81.0
Step 16, Loss: 79.5
Step 17, Loss: 109.0
Step 18, Loss: 235.0
Step 19, Loss: 106.0
Step 20, Loss: 111.0
Step 21, Loss: 376.0
Step 22, Loss: 226.0
Step 23, Loss: 290.0
Step 24, Loss: 119.0
Step 25, Loss: 490.0
Step 26, Loss: 272.0
Step 27, Loss: 225.0
Step 28, Loss: 148.0
Step 29, Loss: 472.0
Step 30, Loss: 107.5
Step 31, Loss: 117.5
Step 32, Loss: 91.0
Step 33, Loss: 255.0
Step 34, Loss: 148.0
Step 35, Loss: 288.0
Step 36, Loss: 206.0
Step 37, Loss: 416.0
Step 38, Loss: 446.0
Step 39, Loss: 320.0
Step 40, Loss: 326.0
Step 41, Loss: 692.0
Step 42, Loss: 520.0
Step 43, Loss: 836.0
Step 44, Loss: 492.0
Step 45, Loss: 338.0
Step 46, Loss: 344.0
Step 47, Loss: 576.0
Step 4

(tensor(402., dtype=torch.bfloat16, grad_fn=<MseLossBackward0>),
 tensor(400., dtype=torch.bfloat16, grad_fn=<DivBackward0>))

In [33]:
batch_size = output.size(0)
batch_idx = random.randint(0, batch_size - 1)
last_pred = output[batch_idx, -2:, :].view(-1).detach().cpu().float().numpy()
last_target = batch["targets"][batch_idx, -2:, :].view(-1).detach().cpu().float().numpy()

fig = px.line()
fig.add_scatter(y=last_pred, mode="lines+markers", name="Prediction")
fig.add_scatter(y=last_target, mode="lines+markers", name="Target")
fig.show()

In [32]:
output[batch_idx, -2:].shape

torch.Size([2, 128])